In [1]:
# STEP 1: Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

In [2]:
# STEP 2: Load Dataset

df = pd.read_csv("logistic_loan_approve.csv")

df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001015,Male,Yes,0,Graduate,No,5720,0,110.0,360.0,1.0,Urban,1
1,LP001022,Male,Yes,1,Graduate,No,3076,1500,126.0,360.0,1.0,Urban,1
2,LP001031,Male,Yes,2,Graduate,No,5000,1800,208.0,360.0,1.0,Urban,1
3,LP001035,Male,Yes,2,Graduate,No,2340,2546,100.0,360.0,NaN,Urban,0
4,LP001051,Male,No,0,Not Graduate,No,3276,0,78.0,360.0,1.0,Urban,1


In [3]:
# STEP 3: Handle Missing Values (VERY IMPORTANT)

df['Gender'].fillna(df['Gender'].mode()[0], inplace = True)
df['Married'].fillna(df['Married'].mode()[0], inplace = True)
df['Dependents'].fillna(df['Dependents'].mode()[0], inplace = True)
df['Self_Employed'].fillna(df['Self_Employed'].mode()[0], inplace = True)

df['LoanAmount'].fillna(df['LoanAmount'].mode()[0], inplace = True)
df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0], inplace = True)
df['Credit_History'].fillna(df['Credit_History'].mode()[0], inplace = True)
df['Loan_Status'].fillna(df['Loan_Status'].mode()[0], inplace = True)


In [4]:
df.fillna({
    'LoanAmount': df['LoanAmount'].median(),
    'Loan_Amount_Term': df['Loan_Amount_Term'].median(),
    'Credit_History': df['Credit_History'].mode()[0]
}, inplace=True)

In [5]:
df.isnull().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [6]:
# Check Infinity Values (ADVANCED BUT IMPORTANT)

import numpy as np

numeric_df = df.select_dtypes(include=[np.number])

print(np.isinf(numeric_df).sum())

ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Loan_Status          0
dtype: int64


In [7]:
# STEP 4: Convert Target Variable (robust to both Y/N and 0/1 datasets)

# If Loan_Status is Y/N, map to 1/0; if it's already numeric (0/1), keep it.
if df['Loan_Status'].dtype == 'O':
    df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
else:
    df['Loan_Status'] = pd.to_numeric(df['Loan_Status'], errors='coerce')

# Safety: ensure only 0/1 and no missing values remain
df['Loan_Status'] = df['Loan_Status'].fillna(df['Loan_Status'].mode()[0]).astype(int)

In [8]:
# STEP 5: Drop Useless Column

df = df.drop('Loan_ID', axis=1)

In [9]:
# STEP 6: Convert Categorical → Numeric

df = pd.get_dummies(df, drop_first=True)

# 👉 Converts:

# Gender → Male/Female
# Property_Area → Urban/Rural

In [10]:
# STEP 7: Define Features & Target

X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

In [11]:
# STEP 8: Train-Test-Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# X_train.to_string()

In [12]:
# STEP 9: Feature Scaling
# Make sure all features are numeric floats (some sklearn builds can be strict with bool dummies)
X_train = X_train.astype(float)
X_test = X_test.astype(float)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [13]:
# STEP 10: Train Logistic Regression Model

model = LogisticRegression(max_iter=5000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=5000)

In [14]:
# STEP 11: MODEL PREDICTION (MAIN PART)

# Class Prediction
y_pred = model.predict(X_test_scaled)
y_pred

array([0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0])

In [15]:
# Probability Prediction

y_prob = model.predict_proba(X_test_scaled)[:, 1]

# 👉 Example:
# [0.82, 0.34, 0.65]

In [16]:
# STEP 12: Evaluation

# Accuracy

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7837837837837838


In [17]:
# Confusion Matrix

print(confusion_matrix(y_test, y_pred))

[[26 13]
 [ 3 32]]


In [18]:
# Classification Report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.67      0.76        39
           1       0.71      0.91      0.80        35

    accuracy                           0.78        74
   macro avg       0.80      0.79      0.78        74
weighted avg       0.81      0.78      0.78        74



In [19]:
# AUC Score

print("AUC:", roc_auc_score(y_test, y_prob))

AUC: 0.9318681318681319


In [20]:
# STEP 13: Predict NEW LOAN APPLICATION

# X_test is a DataFrame; use iloc to grab a single row as a 2D frame.
sample = X_test.iloc[0:1].astype(float)
sample_scaled = scaler.transform(sample)

pred = model.predict(sample_scaled)
prob = model.predict_proba(sample_scaled)

print("Loan Approval Prediction:", pred)
print("Probability:", prob)

Loan Approval Prediction: [0]
Probability: [[0.99566443 0.00433557]]
